In [77]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
import statsmodels.stats.api as sms
from scipy.stats import norm
from scipy import stats
from pathlib import Path
# Disable Jupyter output truncation completely
from IPython.display import display
import sys
# Import necessary modules
from scipy import stats as sp_stats
from scipy.stats import gaussian_kde
# Calculate Pearson correlation with p-values
from scipy.stats import pearsonr

# Display with better formatting - all columns visible
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

In [78]:
# Gets the parent directory of the current working directory
project_root = Path.cwd().parent.parent
fullpath = "{}/data/VST152/pva_donors.csv".format(project_root)
donor_data = pd.read_csv(fullpath)

# Simple Linear Regression Analysis - Donation Amount

## Overview

This notebook performs a comprehensive simple linear regression analysis to understand the relationship between **Donation_Amt** (dependent variable) and **GiftAvgCard36** (independent variable).

### Analysis Components

1. **Data Summary Tables**
   - Observations read, used, and missing
   - Data quality assessment

2. **Analysis of Variance (ANOVA)**
   - Model fit assessment
   - Statistical significance of the regression

3. **Model Fit Statistics**
   - R-Square and Adjusted R-Square
   - Root Mean Square Error (RMSE)
   - Coefficient of Variation

4. **Parameter Estimates**
   - Intercept and slope coefficients
   - Standard errors and t-values
   - 95% confidence intervals
   - Statistical significance testing

5. **Diagnostic Plots**
   - Residuals vs Predicted values
   - Q-Q plot for normality
   - Scale-Location plot
   - Residuals vs Leverage

6. **Prediction Plots**
   - Observed vs Predicted values
   - Scatter plot with regression line and confidence bands

### Research Question

**Does the average credit card gift amount (GiftAvgCard36) significantly predict the donation amount?**

### Variables

- **Dependent Variable (Y)**: Donation_Amt (Donation Amount in dollars)
- **Independent Variable (X)**: GiftAvgCard36 (Average Credit Card Gift in 36 months)

In [79]:
# Column descriptions
col_descriptions = {
    'GiftAvgCard36': 'Average Credit Card Gift (36 Months)',
    'Donation_Amt': 'Donation Amount'
}

# Prepare data for regression
# Remove rows with missing values in either variable
regression_data = donor_data[['GiftAvgCard36', 'Donation_Amt']].dropna()

# Calculate data summary statistics
total_obs = len(donor_data)
used_obs = len(regression_data)
missing_obs = total_obs - used_obs

print(f"Total observations in dataset: {total_obs:,}")
print(f"Observations used in analysis: {used_obs:,}")
print(f"Missing values: {missing_obs:,}")
print(f"Percentage used: {(used_obs/total_obs)*100:.2f}%")

Total observations in dataset: 9,686
Observations used in analysis: 4,111
Missing values: 5,575
Percentage used: 42.44%


## TABLE 1: DATA SUMMARY

Overview of data availability and completeness for the regression analysis.

In [80]:
# TABLE 1: Data Summary
data_summary = pd.DataFrame({
    'Category': ['Number of Observations Read', 'Number of Observations Used', 'Number of Missing Values'],
    'Count': [total_obs, used_obs, missing_obs]
})

data_summary_display = data_summary.copy()
data_summary_display.index.name = None
display(data_summary_display)

,Category,Count
0,Number of Observations Read,9686
1,Number of Observations Used,4111
2,Number of Missing Values,5575


## TABLE 2: ANALYSIS OF VARIANCE (ANOVA)

Tests whether the regression model is statistically significant overall.

### Why ANOVA?

The ANOVA table answers: **"Does the model explain a significant portion of the variation in Donation_Amt?"**

- **Model**: Variation explained by GiftAvgCard36
- **Error**: Unexplained variation (residuals)
- **Corrected Total**: Total variation in Donation_Amt

### Interpretation

- **F Value**: Ratio of explained to unexplained variance
  - Large F → Model explains significant variation
  - Small F → Model explains little variation
  
- **Pr > F**: P-value for the F-test
  - p < 0.05 → Model is statistically significant
  - p ≥ 0.05 → Model is not statistically significant

## REGRESSION ASSUMPTIONS - COMPREHENSIVE GUIDE

Linear regression relies on 5 key assumptions. Violating these assumptions can make your results unreliable. Let's explore each one in detail.

---

## ASSUMPTION 1: LINEARITY

### What is Linearity?

**Linearity** means the relationship between the independent variable (X) and dependent variable (Y) is **linear** (straight line), not curved or non-linear.

### Visual Explanation

```
LINEAR (Good):              NON-LINEAR (Bad):
Y                           Y
^                           ^
|    •                      |        •
|  •   •                    |      •   •
| •     •                   |    •       •
|•       •                  |  •           •
+---------> X              +---------> X
Straight line              Curved pattern
```

### Why is Linearity Important?

1. **Model Validity**: Linear regression assumes a straight-line relationship
2. **Prediction Accuracy**: If relationship is curved, predictions will be systematically wrong
3. **Coefficient Interpretation**: Slope only makes sense if relationship is linear

### Real-World Example: Donation Prediction

**Linear scenario:**
- Every $1 increase in credit card gift → $0.50 increase in donation (constant)
- Relationship is predictable and consistent

**Non-linear scenario:**
- $1 increase in small gifts → $0.10 increase in donation
- $1 increase in large gifts → $0.80 increase in donation
- Relationship changes depending on gift size

### How to Detect Non-Linearity

#### Visual Method (Residuals vs Fitted Plot)
- Plot: Residuals vs Fitted Values
- **Linear**: Points scattered randomly around zero line
- **Non-linear**: Points form a curved pattern (U-shape, inverted U, S-shape)

#### Statistical Tests
- **RESET Test**: Tests for omitted variables and non-linearity
- **Ramsey Test**: Detects specification errors
- **Loess Smoothing**: Visual smoothed curve through residuals

### What to Do If You Find Non-Linearity

1. **Transformation**
   - Log-transform X or Y
   - Polynomial terms (X², X³)
   - Square-root transformation

2. **Polynomial Regression**
   - Add X² term: Y = β₀ + β₁X + β₂X²
   - Allows curved relationships

3. **Spline Regression**
   - Fits different curves in different regions
   - More flexible than polynomial

4. **Non-parametric Methods**
   - LOESS regression
   - Kernel regression
   - Random forests

### In Our Analysis

The **Residuals vs Fitted Values Plot** (top-left in diagnostic plots) shows:
- **X-axis**: Fitted (predicted) values
- **Y-axis**: Residuals (actual - predicted)
- **Pattern**: Should be random scatter around zero line

If you see:
- ✅ **Random scatter**: Linearity assumption is met
- ❌ **U-shaped pattern**: Non-linear relationship (quadratic)
- ❌ **S-shaped pattern**: Non-linear relationship (cubic)
- ❌ **Systematic curve**: Relationship is not linear

---

## ASSUMPTION 2: NORMALITY

### What is Normality?

**Normality** means the **residuals (prediction errors) are normally distributed** (bell-shaped curve).

### Visual Explanation

```
NORMAL DISTRIBUTION:        NON-NORMAL DISTRIBUTION:
    |                           |
    |      •                    |    •
    |    •   •                  |  •   •
    |  •       •                | •     •
    |•           •              |•       •
    +-----------> Residuals    +-----------> Residuals
    Bell-shaped curve          Skewed or heavy-tailed
```

### Why is Normality Important?

1. **Valid Confidence Intervals**: CI calculations assume normality
2. **Valid P-values**: Hypothesis tests rely on normal distribution
3. **Prediction Intervals**: Prediction bands assume normality
4. **Outlier Detection**: Normality helps identify unusual observations

### Real-World Example: Donation Prediction

**Normal residuals:**
- Most predictions are close to actual values
- Few predictions are very far off
- Errors are symmetric (equally likely to be too high or too low)

**Non-normal residuals:**
- Many predictions are slightly off
- Few predictions are extremely far off
- Errors are skewed (more often too high than too low)

### How to Detect Non-Normality

#### Visual Method (Q-Q Plot)
- Plot: Theoretical quantiles vs Sample quantiles
- **Normal**: Points follow the diagonal line
- **Non-normal**: Points deviate from the line

Patterns indicate:
- **S-shaped**: Heavy tails (more extreme values than normal)
- **Curved up**: Right-skewed (tail on right)
- **Curved down**: Left-skewed (tail on left)

#### Statistical Tests
- **Shapiro-Wilk Test**: Most powerful for small samples
- **Kolmogorov-Smirnov Test**: General normality test
- **Anderson-Darling Test**: Sensitive to tails
- **Jarque-Bera Test**: Tests skewness and kurtosis

### What to Do If You Find Non-Normality

1. **Transformation**
   - Log-transform Y
   - Box-Cox transformation
   - Square-root transformation

2. **Robust Methods**
   - Quantile regression
   - Robust regression (Huber, bisquare)
   - Bootstrapping

3. **Larger Sample Size**
   - Central Limit Theorem: Large samples approach normality
   - With n > 30, non-normality is less critical

4. **Different Model**
   - Generalized Linear Models (GLM)
   - Non-parametric methods

### In Our Analysis

The **Q-Q Plot** (top-right in diagnostic plots) shows:
- **X-axis**: Theoretical normal quantiles
- **Y-axis**: Sample quantiles (your residuals)
- **Pattern**: Should follow the diagonal line

If you see:
- ✅ **Points on diagonal line**: Normality assumption is met
- ❌ **S-shaped curve**: Heavy tails (leptokurtic)
- ❌ **Curved up**: Right-skewed residuals
- ❌ **Curved down**: Left-skewed residuals
- ❌ **Points far from line at ends**: Outliers present

---

## ASSUMPTION 3: HOMOSCEDASTICITY

### What is Homoscedasticity?

**Homoscedasticity** (also called "homogeneity of variance") means that the **variance of residuals is constant across all predicted values**.

In simpler terms: **The spread of prediction errors should be the same whether predictions are small or large.**

### Visual Explanation

```
HOMOSCEDASTIC (Good - Constant Variance):
Y
^
|     •  •  •
|   •  •  •  •
| •  •  •  •  •
|   •  •  •  •
|     •  •  •
+-------------------> X
Errors spread equally at all X values

HETEROSCEDASTIC (Bad - Non-Constant Variance):
Y
^
|                    •
|                  •   •
|                •   •   •
|   •  •  •    •   •   •
| •  •  •  •  •   •   •
+-------------------> X
Errors spread more at higher X values (funnel shape)
```

### Why is Homoscedasticity Important?

1. **Validity of Statistical Tests**
   - Linear regression assumes constant variance
   - If variance changes, p-values and confidence intervals become **unreliable**
   - You might think a coefficient is significant when it's not (or vice versa)

2. **Prediction Accuracy**
   - With homoscedasticity: Prediction errors are consistent
   - With heteroscedasticity: Predictions are more accurate for some ranges than others
   - Example: Model predicts small donations well but poorly predicts large donations

3. **Efficiency of Estimates**
   - Homoscedasticity: Standard errors are minimized
   - Heteroscedasticity: Standard errors are inflated, making estimates less precise

### Real-World Example: Donation Prediction

**Homoscedastic scenario:**
- When predicting $10 donations: Error ≈ ±$5
- When predicting $100 donations: Error ≈ ±$5
- Consistent prediction accuracy across all donation amounts

**Heteroscedastic scenario:**
- When predicting $10 donations: Error ≈ ±$2
- When predicting $100 donations: Error ≈ ±$50
- Model is much better at predicting small donations than large ones

### How to Detect Heteroscedasticity

#### Visual Method (Scale-Location Plot)
- Plot: √|Standardized Residuals| vs Fitted Values
- **Homoscedastic**: Points scattered randomly in a horizontal band
- **Heteroscedastic**: Points form a pattern (funnel, cone, or curve)

#### Statistical Tests
- **Breusch-Pagan Test**: Tests if variance depends on X
- **White Test**: More general test for heteroscedasticity
- **Goldfeld-Quandt Test**: Compares variance in two subsamples

### What to Do If You Find Heteroscedasticity

1. **Transformation**
   - Log-transform the dependent variable
   - Square-root transformation
   - Box-Cox transformation

2. **Weighted Least Squares (WLS)**
   - Give less weight to observations with higher variance
   - More sophisticated than OLS

3. **Robust Standard Errors**
   - Use Huber-White standard errors
   - Adjusts confidence intervals without changing coefficients

4. **Different Model**
   - Try non-linear models
   - Use generalized linear models (GLM)

### In Our Analysis

The **Scale-Location Plot** (bottom-left in diagnostic plots) shows:
- **X-axis**: Fitted (predicted) values
- **Y-axis**: √|Standardized Residuals|
- **Pattern**: Should be random scatter (no trend)

If you see:
- ✅ **Random scatter**: Homoscedasticity assumption is met
- ❌ **Upward trend**: Variance increases with predictions (heteroscedastic)
- ❌ **Downward trend**: Variance decreases with predictions (heteroscedastic)
- ❌ **Funnel shape**: Classic heteroscedasticity pattern

---

## ASSUMPTION 4: INDEPENDENCE

### What is Independence?

**Independence** means the **residuals are not correlated with each other**. Each observation's error is independent of other observations' errors.

### Visual Explanation

```
INDEPENDENT (Good):         DEPENDENT (Bad):
Residuals                   Residuals
^                           ^
|  •  •  •  •              |  •  •  •  •
|    •  •  •  •            |    •  •  •  •
|  •  •  •  •              |  •  •  •  •
|    •  •  •  •            |    •  •  •  •
+---------> Time           +---------> Time
Random scatter             Systematic pattern
```

### Why is Independence Important?

1. **Valid Standard Errors**: Correlated errors inflate standard errors
2. **Overstated Significance**: P-values become too small (false positives)
3. **Inefficient Estimates**: Confidence intervals are too narrow
4. **Prediction Accuracy**: Correlated errors reduce prediction reliability

### Real-World Example: Donation Prediction

**Independent scenario:**
- If model overestimates one donor's gift, it doesn't affect prediction for next donor
- Errors are random and unpredictable

**Dependent scenario (Time Series):**
- If model overestimates one month's donations, it tends to overestimate next month too
- Errors are correlated over time (autocorrelation)

**Dependent scenario (Spatial):**
- If model overestimates donations in one region, it tends to overestimate nearby regions
- Errors are correlated across space

### How to Detect Dependence

#### Visual Method (Residuals vs Order Plot)
- Plot: Residuals in order they were collected
- **Independent**: Random scatter
- **Dependent**: Systematic pattern or trend

#### Statistical Tests
- **Durbin-Watson Test**: Detects autocorrelation (time series)
- **Moran's I**: Detects spatial autocorrelation
- **Ljung-Box Test**: Tests for autocorrelation at multiple lags
- **ACF/PACF Plots**: Visual autocorrelation plots

### What to Do If You Find Dependence

1. **Time Series Methods**
   - ARIMA models
   - GARCH models
   - Vector autoregression (VAR)

2. **Spatial Methods**
   - Spatial regression
   - Geographically weighted regression
   - Spatial lag models

3. **Clustering Methods**
   - Mixed-effects models
   - Hierarchical models
   - Multilevel models

4. **Robust Methods**
   - Cluster-robust standard errors
   - Newey-West standard errors

### In Our Analysis

The **Residuals vs Order Plot** (not shown but important) would show:
- **X-axis**: Order of observations
- **Y-axis**: Residuals
- **Pattern**: Should be random scatter

If you see:
- ✅ **Random scatter**: Independence assumption is met
- ❌ **Upward/downward trend**: Systematic pattern (non-independence)
- ❌ **Cyclical pattern**: Periodic dependence
- ❌ **Clustering**: Groups of similar residuals

---

## ASSUMPTION 5: NO OUTLIERS / NO INFLUENTIAL POINTS

### What are Outliers and Influential Points?

**Outliers**: Observations with unusual Y values (very large residuals)
**Influential Points**: Observations that disproportionately affect the regression line

An observation can be:
- Outlier but not influential
- Influential but not outlier
- Both outlier and influential (most problematic)

### Visual Explanation

```
NO OUTLIERS (Good):         WITH OUTLIERS (Bad):
Y                           Y
^                           ^
|    •  •  •               |    •  •  •
|  •  •  •  •              |  •  •  •  •
| •  •  •  •  •            | •  •  •  •  •
|  •  •  •  •              |  •  •  •  •
|    •  •  •               |    •  •  •
+---------> X              |              •  (Outlier)
All points close to line   +---------> X
```

### Why are Outliers Important?

1. **Biased Estimates**: Outliers can pull the regression line away from the true relationship
2. **Inflated Errors**: Outliers increase residual sum of squares
3. **Reduced Precision**: Standard errors become larger
4. **Invalid Conclusions**: Outliers can reverse the sign of a coefficient

### Real-World Example: Donation Prediction

**No outliers:**
- Donations range from $0 to $200
- Model fits well for all donors
- Predictions are reliable

**With outliers:**
- Most donations $0-$200, but one donor gave $10,000
- Regression line is pulled toward the outlier
- Model fits poorly for typical donors
- Predictions are unreliable

### How to Detect Outliers and Influential Points

#### Visual Method (Residuals vs Leverage Plot)
- Plot: Standardized Residuals vs Leverage
- **Leverage**: How far X is from the mean
- **Residual**: How far Y is from the fitted line

#### Statistical Measures

**Leverage (Hat Values)**
- Measures how unusual X is
- Range: 0 to 1
- High leverage: Observation is far from mean of X

**Standardized Residuals**
- Residuals divided by standard error
- |Residual| > 2: Potential outlier
- |Residual| > 3: Likely outlier

**Cook's Distance**
- Measures influence on regression coefficients
- Cook's D > 4/n: Potentially influential
- Cook's D > 1: Definitely influential

**DFBETA**
- Change in coefficient if observation removed
- |DFBETA| > 2/√n: Potentially influential

### What to Do If You Find Outliers

1. **Investigate**
   - Is it a data entry error?
   - Is it a real but unusual observation?
   - Should it be included in analysis?

2. **Remove**
   - Delete the outlier if it's an error
   - Document why it was removed

3. **Transform**
   - Log-transform to reduce impact
   - Box-Cox transformation

4. **Robust Methods**
   - Robust regression (Huber, bisquare)
   - Quantile regression
   - Winsorization (cap extreme values)

5. **Separate Analysis**
   - Analyze with and without outliers
   - Report both results

### In Our Analysis

The **Residuals vs Leverage Plot** (bottom-right in diagnostic plots) shows:
- **X-axis**: Leverage (how unusual X is)
- **Y-axis**: Standardized Residuals (how unusual Y is)
- **Pattern**: Most points should be in the middle

If you see:
- ✅ **Points clustered in middle**: No problematic outliers
- ❌ **Points far right with large residuals**: Influential outliers
- ❌ **Points far right with small residuals**: High leverage but not outlier
- ❌ **Points far from zero line**: Outliers (but may not be influential)

## DIAGNOSTIC PLOTS

Diagnostic plots assess the validity of regression assumptions.

### Assumptions Being Tested

1. **Linearity**: Relationship between X and Y is linear
2. **Normality**: Residuals are normally distributed
3. **Homoscedasticity**: Constant variance of residuals
4. **Independence**: Residuals are independent
5. **No Outliers**: No extreme values unduly influencing the model

## HOW TO INTERPRET DIAGNOSTIC PLOTS - PRACTICAL GUIDE

After seeing the diagnostic plots, here's how to draw conclusions and make judgments about your regression model.

---

### PLOT 1: RESIDUALS VS FITTED VALUES (Top-Left)

**What it shows:**
- X-axis: Predicted donation amounts
- Y-axis: Prediction errors (actual - predicted)
- Blue dots: Individual predictions
- Red dashed line: Perfect predictions (error = 0)
- Green line: Trend of errors across predictions

**What to look for:**

#### ✅ GOOD SIGNS (Model is valid):
1. **Random scatter around zero line**
   - Points are randomly distributed above and below red line
   - No systematic pattern
   - Conclusion: Linearity assumption is met ✓

2. **Constant spread across all predictions**
   - Errors are equally spread at low, medium, and high predictions
   - No funnel or cone shape
   - Conclusion: Homoscedasticity assumption is met ✓

3. **Green trend line is flat**
   - Trend line stays close to zero
   - No upward or downward slope
   - Conclusion: No systematic bias in predictions ✓

#### ❌ RED FLAGS (Model has problems):
1. **U-shaped or curved pattern**
   - Points form a curve (not random)
   - Conclusion: Relationship is non-linear, need transformation or polynomial terms

2. **Funnel shape (wider on right)**
   - Errors increase with predictions
   - Conclusion: Heteroscedasticity - use weighted regression or transform Y

3. **Funnel shape (wider on left)**
   - Errors decrease with predictions
   - Conclusion: Heteroscedasticity - use weighted regression or transform Y

4. **Green trend line slopes up or down**
   - Systematic bias in predictions
   - Conclusion: Model systematically over/under-predicts at certain ranges

**Example Interpretation:**
- If you see random scatter with flat trend line → Model is good for linearity and homoscedasticity
- If you see U-shape → Try adding GiftAvgCard36² term to model
- If you see funnel → Try log-transforming Donation_Amt

---

### PLOT 2: NORMAL Q-Q PLOT (Top-Right)

**What it shows:**
- X-axis: Theoretical normal distribution quantiles
- Y-axis: Your actual residual quantiles
- Blue dots: Your residuals
- Red line: Perfect normal distribution

**What to look for:**

#### ✅ GOOD SIGNS (Residuals are normal):
1. **Points follow the red line closely**
   - Most points lie on or very close to the diagonal
   - Conclusion: Residuals are normally distributed ✓

2. **Slight deviations only at the ends**
   - Points may deviate slightly at extreme values
   - But most of the line is followed
   - Conclusion: Normality assumption is reasonably met ✓

3. **Symmetric deviations**
   - Points deviate equally above and below the line
   - No consistent pattern
   - Conclusion: No systematic skewness ✓

#### ❌ RED FLAGS (Residuals are not normal):
1. **S-shaped curve**
   - Points curve up at both ends
   - Conclusion: Heavy tails (more extreme values than normal) - use robust methods

2. **Curved up (like a smile)**
   - Points curve upward
   - Conclusion: Right-skewed residuals - try log-transform Y

3. **Curved down (like a frown)**
   - Points curve downward
   - Conclusion: Left-skewed residuals - try square-root or Box-Cox transform

4. **Points far from line at ends**
   - Large deviations at extreme values
   - Conclusion: Outliers present - investigate and possibly remove

**Example Interpretation:**
- If points follow red line → Normality assumption is met ✓
- If S-shaped curve → Use robust regression or larger sample
- If curved up → Try log-transforming Donation_Amt

---

### PLOT 3: SCALE-LOCATION PLOT (Bottom-Left)

**What it shows:**
- X-axis: Predicted donation amounts
- Y-axis: Square root of standardized residuals (spread of errors)
- Blue dots: Individual predictions
- Green line: Trend of error spread
- Orange line: Expected level if homoscedastic

**What to look for:**

#### ✅ GOOD SIGNS (Constant variance):
1. **Points scattered randomly in horizontal band**
   - No systematic pattern
   - Spread is similar across all predictions
   - Conclusion: Homoscedasticity assumption is met ✓

2. **Green trend line is flat**
   - Stays close to orange reference line
   - No upward or downward slope
   - Conclusion: Variance is constant ✓

3. **Similar vertical spread at all X values**
   - Low predictions: Points spread equally
   - High predictions: Points spread equally
   - Conclusion: Prediction accuracy is consistent ✓

#### ❌ RED FLAGS (Non-constant variance):
1. **Upward trend (funnel opens right)**
   - Green line slopes upward
   - Errors increase with predictions
   - Conclusion: Heteroscedasticity - model is less accurate for large predictions

2. **Downward trend (funnel opens left)**
   - Green line slopes downward
   - Errors decrease with predictions
   - Conclusion: Heteroscedasticity - model is less accurate for small predictions

3. **Cone or megaphone shape**
   - Spread increases or decreases systematically
   - Conclusion: Variance depends on prediction size - use weighted regression

**Example Interpretation:**
- If flat green line with random scatter → Homoscedasticity is met ✓
- If upward trend → Donations are harder to predict at higher amounts
- If downward trend → Donations are harder to predict at lower amounts

---

### PLOT 4: RESIDUALS VS LEVERAGE (Bottom-Right)

**What it shows:**
- X-axis: Leverage (how unusual the X value is)
- Y-axis: Standardized residuals (how unusual the Y value is)
- Blue dots: Individual observations
- Orange dashed lines: ±2 standard deviations (potential outliers)
- Red dashed lines: ±3 standard deviations (likely outliers)
- Red circles: Influential points (Cook's D > threshold)

**What to look for:**

#### ✅ GOOD SIGNS (No problematic outliers):
1. **Most points between ±2 lines**
   - Points clustered in middle region
   - Few or no points beyond ±2 or ±3 lines
   - Conclusion: No significant outliers ✓

2. **No red circles**
   - No influential points highlighted
   - Conclusion: No observations unduly affecting the model ✓

3. **Points in lower-left region**
   - Low leverage (typical X values)
   - Small residuals (good predictions)
   - Conclusion: Model fits well for typical observations ✓

#### ❌ RED FLAGS (Problematic outliers):
1. **Points beyond ±3 lines**
   - Extreme residuals
   - Conclusion: Outliers present - investigate and possibly remove

2. **Red circles (influential points)**
   - Points with high Cook's D
   - Conclusion: These observations disproportionately affect the model

3. **Points far right with large residuals**
   - High leverage AND large residuals
   - Conclusion: Influential outliers - most problematic type

4. **Points far right with small residuals**
   - High leverage but good predictions
   - Conclusion: Unusual X values but model handles them well

**Example Interpretation:**
- If all points between ±2 lines with no red circles → No problematic outliers ✓
- If red circles present → Investigate those donors (data entry error? unusual behavior?)
- If points beyond ±3 lines → Consider removing or using robust regression

---

## OVERALL MODEL ASSESSMENT CHECKLIST

After reviewing all 4 plots, use this checklist:

| Assumption | Plot | Good Sign | Bad Sign | Action |
|-----------|------|-----------|----------|--------|
| **Linearity** | Plot 1 | Random scatter, flat trend | U-shape or curve | Add polynomial terms |
| **Normality** | Plot 2 | Points on red line | S-shape or curve | Transform Y or use robust methods |
| **Homoscedasticity** | Plot 3 | Flat green line, random scatter | Upward/downward trend | Use weighted regression or transform |
| **Independence** | (Not shown) | Random order | Systematic pattern | Use time series or spatial methods |
| **No Outliers** | Plot 4 | Points between ±2, no red circles | Points beyond ±3 or red circles | Investigate and possibly remove |

---

## DECISION TREE: WHAT TO DO BASED ON PLOTS

```
START: Review all 4 diagnostic plots
│
├─ All plots look good?
│  └─ YES → Model is valid! Use it for predictions ✓
│
├─ Plot 1 shows U-shape?
│  └─ YES → Add polynomial term (X²) or try log-transform
│
├─ Plot 2 shows S-curve?
│  └─ YES → Use robust regression or larger sample
│
├─ Plot 3 shows upward trend?
│  └─ YES → Use weighted regression or log-transform Y
│
├─ Plot 4 shows red circles?
│  └─ YES → Investigate those observations
│           ├─ Data error? → Remove
│           ├─ Real but unusual? → Keep or use robust regression
│           └─ Legitimate outlier? → Report separately
│
└─ Multiple problems?
   └─ YES → Consider different model (GLM, non-parametric, etc.)
```

---

## REAL-WORLD EXAMPLE: INTERPRETING YOUR DONATION MODEL

**Scenario 1: All plots look good**
- Conclusion: GiftAvgCard36 is a good linear predictor of Donation_Amt
- Action: Use the model for predictions and decision-making
- Confidence: High

**Scenario 2: Plot 1 shows funnel (wider on right)**
- Conclusion: Model predicts small donations well but struggles with large donations
- Action: Use weighted regression or log-transform Donation_Amt
- Confidence: Medium - need to fix heteroscedasticity

**Scenario 3: Plot 4 shows 5 red circles**
- Conclusion: 5 donors have unusual donation patterns
- Action: Investigate these donors
  - Are they major donors? (Keep them)
  - Data entry errors? (Remove them)
  - Unusual circumstances? (Document them)
- Confidence: Low - need to understand outliers

**Scenario 4: Plot 2 shows S-curve**
- Conclusion: Residuals have heavier tails than normal
- Action: Use robust regression or bootstrap confidence intervals
- Confidence: Medium - normality is less critical with large samples

## SCATTER PLOT WITH REGRESSION LINE

Visualization of the relationship between GiftAvgCard36 and Donation_Amt with regression line and 95% confidence bands.

## OBSERVED VS PREDICTED VALUES PLOT

Compares actual donation amounts with model predictions to assess prediction accuracy.